In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import cm, colors
import glob

import hoomd
import gc
import warnings
import copy
import pathlib

import os
import shutil

import signac

In [2]:
from monk import nb, prep, pair, render, utils, workflow, grid

parent = pathlib.Path(os.getcwd()).parent / "config.yaml"
config = workflow.get_config(parent.as_posix())
parent, config

(PosixPath('/home/ian/Projects/work/monk/workflows/hierarchy/config.yaml'),
 {'root': '/media/ian/Data2/monk/hierarchy',
  'origin': '/media/ian/Data2/monk/2d-esl'})

In [3]:
project: signac.Project = signac.init_project(name="hierarchy", root=config['root'])
project.doc

{}

In [4]:
for job in project:
    print(job)

In [5]:
origin: signac.Project = signac.get_project(root=config['origin'])
origin.doc

{'avail_seed': 11, 'dt': 0.005, 'step_unit': 1000, 'temp_steps': 20, 'equil_time': 40, 'run_time': 10, 'max_alpha_time': 1000, 'alpha_iters': 10, '_status': {}}

In [28]:
tgs = {}
for job in origin:
    try:
        tg = job.data["results/tg"]
        if job.sp["replica"] == 0:
            tgs[job.sp["pot"]] = tg
            print(tg)
    except KeyError:
        pass
tgs

0.19836457263395846
0.12345081002164043


{'KA_LJ': 0.19836457263395846, 'KA_WCA': 0.12345081002164043}

In [29]:
# only set once
if "avail_seed" not in project.doc:
    project.doc["avail_seed"] = 0

In [30]:
project.doc["avail_seed"]

0

In [31]:
project.doc["dt"] = 0.005
project.doc["step_unit"] = 200
project.doc["equil_time"] = 100
project.doc["min_periods"] = 100
project.doc["dumps"] = 20
project.doc["period_times"] = [1000.0]
project.doc["max_shears"] = [0.04, 0.045, 0.05, 0.055, 0.06]

In [39]:
# fix init_temp
for job in project:
    print(job)
    print(job.sp)
    temp = float(job.sp["prep"].split("_")[1])
    print(job.doc)
    # job.doc["init_temp"] = temp

d6c3d182e26626c19125d78ce7e0dbca
{'N': 32768, 'replica': 8, 'rho': 1.2, 'pot': 'KA_LJ', 'prep': 'GQ_0.600'}
{'init': True, 'seed': 63, 'init_temp': 0.6, 'tg': 0.19836457263395846, 'temps': [0.0019836457263395844]}
85e104d1441afef9dd81559a07032576
{'N': 32768, 'replica': 0, 'rho': 1.2, 'pot': 'KA_LJ', 'prep': 'GQ_0.375'}
{'init': True, 'seed': 2, 'init_temp': 0.375, 'tg': 0.19836457263395846, 'temps': [0.0019836457263395844]}
03445d5feb9254395686a7f2f5f4e69b
{'N': 32768, 'replica': 7, 'rho': 1.2, 'pot': 'KA_LJ', 'prep': 'GQ_0.637'}
{'init': True, 'seed': 53, 'init_temp': 0.637, 'tg': 0.19836457263395846, 'temps': [0.0019836457263395844]}
ee07fd04c800b775d7ad665ee7c33df0
{'N': 32768, 'replica': 4, 'rho': 1.2, 'pot': 'KA_LJ', 'prep': 'GQ_0.525'}
{'init': True, 'seed': 28, 'init_temp': 0.525, 'tg': 0.19836457263395846, 'temps': [0.0019836457263395844]}
841f5e51053c13e5968bc007ea034a86
{'N': 32768, 'replica': 8, 'rho': 1.2, 'pot': 'KA_LJ', 'prep': 'GQ_0.300'}
{'init': True, 'seed': 55, 'ini

In [10]:
# HTL

N = 2**15  # 32758
rhos = [1.2]
init_temps = [1.5]
pot = "KA_LJ"
prepared = "HTL"

# for replica in range(6):
#     for init_temp, rho in zip(init_temps, rhos):
#         sp = {"N": N, "replica": replica, "rho": rho, "pot": pot, "prep": prepared}
#         job = project.open_job(sp)
#         if "init" not in job.doc:
#             tg = tgs[pot]
#             job.init()
#             job.doc["init"] = True
#             job.doc["seed"] = project.doc["avail_seed"]
#             job.doc["init_temp"] = init_temp
#             job.doc["tg"] = tg
#             job.doc["temps"] = [tg/4.0, tg/2.0, tg*3.0/4.0, tg]
#             project.doc["avail_seed"] += 1

/home/ian/Projects/work/monk/.venv/lib/python3.10/site-packages/signac/synced_collections/numpy_utils.py:46: NumpyConversionWarning: Any numpy types provided will be transparently converted to the closest base Python equivalents.
  warnings.warn(NUMPY_CONVERSION_WARNING, NumpyConversionWarning)


In [29]:
# for job in project.find_jobs({"prep": "ESL"}):
#     print(job, job.sp)
#     job.remove()

In [16]:
pot = "KA_LJ"
for replica in [0]:
    matching_jobs = origin.find_jobs({"pot": pot, "replica": replica})
    assert len(matching_jobs) == 1
    origin_job = matching_jobs.next()
    sims = sorted(glob.glob(origin_job.fn("fine/equil_*.gsd")))
    sims = [sim.split("/equil_temp-")[-1].split(".gsd")[0] for sim in sims]
    print(len(sims))
    print(sims)

11
['0.300', '0.337', '0.375', '0.412', '0.450', '0.487', '0.525', '0.562', '0.600', '0.637', '0.675']


In [33]:
# ESL

N = 2**15  # 32758
rhos = [1.2]
pot = "KA_LJ"
replicas = [0, 2, 4, 5, 7, 8]

for replica in replicas:
    matching_jobs = origin.find_jobs({"pot": pot, "replica": replica})
    assert len(matching_jobs) == 1
    origin_job = matching_jobs.next()
    sims = sorted(glob.glob(origin_job.fn("fine/equil_*.gsd")))
    temps = [sim.split("/equil_temp-")[-1].split(".gsd")[0] for sim in sims]
    # print(sim)
    # assert float(utils.extract_between(sim, "equil_temp-", ".gsd")) == 0.3
    for sim, temp in zip(sims, temps):
        prep = f"GQ_{temp}"
        print(prep)
        for rho in rhos:
            sp = {"N": N, "replica": replica, "rho": rho, "pot": pot, "prep": prep}
            job = project.open_job(sp)
            if "init" not in job.doc:
                tg = tgs[pot]
                job.init()
                job.doc["init"] = True
                job.doc["seed"] = project.doc["avail_seed"]
                job.doc["init_temp"] = 0.3
                job.doc["tg"] = tg
                job.doc["temps"] = [tg/100.0]
                project.doc["avail_seed"] += 1
                shutil.copy(sim, job.fn("start-file.gsd"))
                

GQ_0.300


/home/ian/Projects/work/monk/.venv/lib/python3.10/site-packages/signac/synced_collections/numpy_utils.py:46: NumpyConversionWarning: Any numpy types provided will be transparently converted to the closest base Python equivalents.
  warnings.warn(NUMPY_CONVERSION_WARNING, NumpyConversionWarning)


GQ_0.337
GQ_0.375
GQ_0.412
GQ_0.450
GQ_0.487
GQ_0.525
GQ_0.562
GQ_0.600
GQ_0.637
GQ_0.675
GQ_0.300
GQ_0.337
GQ_0.375
GQ_0.412
GQ_0.450
GQ_0.487
GQ_0.525
GQ_0.562
GQ_0.600
GQ_0.637
GQ_0.675
GQ_0.300
GQ_0.337
GQ_0.375
GQ_0.412
GQ_0.450
GQ_0.487
GQ_0.525
GQ_0.562
GQ_0.600
GQ_0.637
GQ_0.675
GQ_0.300
GQ_0.337
GQ_0.375
GQ_0.412
GQ_0.450
GQ_0.487
GQ_0.525
GQ_0.562
GQ_0.600
GQ_0.637
GQ_0.675
GQ_0.300
GQ_0.337
GQ_0.375
GQ_0.412
GQ_0.450
GQ_0.487
GQ_0.525
GQ_0.562
GQ_0.600
GQ_0.637
GQ_0.675
GQ_0.300
GQ_0.337
GQ_0.375
GQ_0.412
GQ_0.450
GQ_0.487
GQ_0.525
GQ_0.562
GQ_0.600
GQ_0.637
GQ_0.675


In [ ]:
# initialize projects using data from the prior project

# some data should be copied, others probably don't need to be